# Check an Analyst report

Copy this file into `private/reviews/` and rename the copy to `<packet_id>_review.ipynb`
(for example, `EP_0003_review.ipynb`). Work in that copy; keep this template blank.

1. Set `attempt_id` below to the folder name printed by generation.
2. Run the cells and read each claim beside its evidence.
3. Fill in **Notes / feedback** and make any fixes in the edit cell.
4. When finished, set `ready = True`, run all cells again, and save the notebook.

The last cell saves the final report to public. The original candidate stays unchanged.
There are no model calls here. Leave `ready = False` while checking.


In [1]:
import copy
import json
import sys
from decimal import Decimal
from pathlib import Path
from pprint import pprint

repo = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "data/benchmark/private/packets").is_dir())
sys.path.insert(0, str(repo))
from src.benchmark.prepare import read, save, mask, assert_masked, check_report
from src.benchmark.schemas import Packet, Report

folder = repo / "data/benchmark"
attempt_id = "a4db15fb4a7644428ea25704e0f5074f"  # change this in your copy
attempt_folder = folder / "private/attempts" / attempt_id
attempt = read(attempt_folder / "attempt.json")
sent_packet = json.loads(attempt["request"]["messages"][1]["content"])["evidence_packet"]
original_report = read(attempt_folder / "candidate.json")
aliases = read(folder / "private/aliases.json")
packet = mask(sent_packet, aliases)
report = copy.deepcopy(original_report)
print("Packet:", packet["packet_id"], "| Model:", attempt["request"]["model"])
print("Packet masking changed:", packet != sent_packet)

Packet: EP_0008 | Model: google/gemini-2.5-flash
Packet masking changed: False


In [2]:
print("Recommendation:", report["recommendation"])
evidence = {e["evidence_id"]: e["text"] for e in packet["evidence"]}
for claim in report["claims"]:
    print("\nCLAIM", claim["claim_id"], claim["text"])
    for evidence_id in claim["evidence_ids"]:
        print("EVIDENCE", evidence_id, evidence.get(evidence_id, "MISSING"))
print("\nRationale:", report["rationale"])

Recommendation: HOLD

CLAIM C1 Total revenues for COMPANY_08 decreased by 2.93% from $97,690 million in 2024 to $94,827 million in 2025.
EVIDENCE EP_0008_E01 Results	of	Operations
Revenues
	
Year	Ended	December	31,
2025	vs.	2024	Change
2024	vs.	2023	Change
(Dollars	in	millions)
2025
2024
2023
$
%
$
%
Automotive	sales
$
65,821	
$
72,480	
$
78,509	
$
(6,659)
(9)
%
$
(6,029)
(8)
%
Automotive	regulatory	credits
1,993	
2,763	
1,790	
(770)
(28)
%
973	
54	
%
Automotive	leasing
1,712	
1,827	
2,120	
(115)
(6)
%
(293)
(14)
%
Total	automotive	revenues
69,526	
77,070	
82,419	
(7,544)
(10)
%
(5,349)
(6)
%
Services	and	other
12,530	
10,534	
8,319	
1,996	
19	
%
2,215	
27	
%
Total	automotive	&	services	and	other	segment
revenue
82,056	
87,604	
90,738	
(5,548)
(6)
%
(3,134)
(3)
%
Energy	generation	and	storage	segment	revenue
12,771	
10,086	
6,035	
2,685	
27	
%
4,051	
67	
%
Total	revenues
$
94,827	
$
97,690	
$
96,773	
$
(2,863)
(3)
%
$
917	
1	
%

CLAIM C2 Automotive revenue, the largest segment, decline

## Numbers

Compare the claimed amount or percentage with the calculation. Also read the prose above:
these checks only cover the structured assertion, not every number in a sentence.

In [3]:
facts_by_id = {f["fact_id"]: f for f in packet["facts"]}
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        print(claim["claim_id"], "Check against the excerpt above.")
        continue
    ids = [spec.get("fact_id")] if spec["kind"] == "numeric" else [spec.get("from_fact_id"), spec.get("to_fact_id")]
    periods = [spec["period"]] if spec["kind"] == "numeric" else [spec["from_period"], spec["to_period"]]
    if all(ids):
        selected = [facts_by_id.get(fact_id) for fact_id in ids]
        print(claim["claim_id"], "Referenced facts:")
        pprint(selected)
        if any(f is None for f in selected):
            print("Unknown fact ID; check manually.")
            continue
    else:
        print(claim["claim_id"], "Older candidate: references are added in the edit cell below.")
        matches = [[f for f in packet["facts"] if f["metric"] == spec["metric"]
                    and f["evidence_id"] in claim["evidence_ids"] and f["period"] == period
                    and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])] for period in periods]
        if any(len(found) != 1 for found in matches):
            print("No exact fact match; check manually.")
            continue
        selected = [found[0] for found in matches]
        pprint(selected)
    values = [Decimal(f["value"]) for f in selected]
    if spec["kind"] == "percentage_change" and values[0] == 0:
        print(claim["claim_id"], "Cannot calculate percentage change from zero.")
        continue
    expected = values[0] if spec["kind"] == "numeric" else (values[1] - values[0]) / values[0] * 100
    print(claim["claim_id"], "claimed:", spec["value"], "calculated:", expected,
          "rounded:", expected.quantize(Decimal("0.01")))
pprint(check_report(Packet.model_validate(packet), Report.model_validate(report)))


C1 Referenced facts:
[{'evidence_id': 'EP_0008_E01',
  'fact_id': 'EP_0008_F02',
  'metric': 'revenue',
  'period': {'end': '2024-12-31', 'label': 'FY2024', 'start': '2024-01-01'},
  'unit': 'USD_million',
  'value': '97690'},
 {'evidence_id': 'EP_0008_E01',
  'fact_id': 'EP_0008_F01',
  'metric': 'revenue',
  'period': {'end': '2025-12-31', 'label': 'FY2025', 'start': '2025-01-01'},
  'unit': 'USD_million',
  'value': '94827'}]
C1 claimed: -2.93 calculated: -2.930699150373630873170232368 rounded: -2.93
C2 Referenced facts:
[{'evidence_id': 'EP_0008_E01',
  'fact_id': 'EP_0008_F04',
  'metric': 'automotive_revenue',
  'period': {'end': '2024-12-31', 'label': 'FY2024', 'start': '2024-01-01'},
  'unit': 'USD_million',
  'value': '77070'},
 {'evidence_id': 'EP_0008_E01',
  'fact_id': 'EP_0008_F03',
  'metric': 'automotive_revenue',
  'period': {'end': '2025-12-31', 'label': 'FY2025', 'start': '2025-01-01'},
  'unit': 'USD_million',
  'value': '69526'}]
C2 claimed: -9.80 calculated: -9.788

## Notes / feedback

Replace the blanks with short notes. Use the evidence-checking notebook to check the source PDF too.

- Source excerpts and masking: ...
- Claims, numbers, years, units and citations: ...
- Reasoning and risks / counter-evidence: ...
- Problems found and what I changed (or no changes needed): C2’s decline was corrected to −9.79% for the automatic check. The sentence keeps the rounded figure of 9.8%. The model’s original output was kept unchanged.
- Decision: checked


In [9]:
# Make any corrections here. Skip this if no changes are needed.
# Example: report["claims"][0]["text"] = "Corrected claim..."
# If you correct a number, update its check_spec too, if present.
# Keep the edits in this cell so running the notebook again applies them.
report["claims"][1]["text"] = report["claims"][1]["text"].replace(
    "9.80%", "9.8%"
)
report["claims"][1]["check_spec"]["value"] = "-9.79"

# Add explicit source-fact references to the reviewed copy of older candidates.
# This does not change the saved model response, numbers, or wording.
for claim in report["claims"]:
    spec = claim["check_spec"]
    if spec is None:
        continue
    fields = [("fact_id", "period")] if spec["kind"] == "numeric" else [("from_fact_id", "from_period"), ("to_fact_id", "to_period")]
    for id_field, period_field in fields:
        if spec.get(id_field):
            continue
        matches = [f for f in packet["facts"] if f["metric"] == spec["metric"]
                   and f["period"] == spec[period_field] and f["evidence_id"] in claim["evidence_ids"]
                   and (spec["kind"] != "numeric" or f["unit"] == spec["unit"])]
        assert len(matches) == 1, "Check the fact mapping manually: " + claim["claim_id"]
        spec[id_field] = matches[0]["fact_id"]

pprint(report, sort_dicts=False)


{'schema_version': '0.1',
 'report_id': 'R_a4db15fb4a7644428ea25704e0f5074f',
 'recommendation': 'HOLD',
 'claims': [{'claim_id': 'C1',
             'text': 'Total revenues for COMPANY_08 decreased by 2.93% from '
                     '$97,690 million in 2024 to $94,827 million in 2025.',
             'evidence_ids': ['EP_0008_E01'],
             'check_spec': {'kind': 'percentage_change',
                            'from_fact_id': 'EP_0008_F02',
                            'to_fact_id': 'EP_0008_F01',
                            'metric': 'revenue',
                            'from_period': {'label': 'FY2024',
                                            'start': '2024-01-01',
                                            'end': '2024-12-31'},
                            'to_period': {'label': 'FY2025',
                                          'start': '2025-01-01',
                                          'end': '2025-12-31'},
                            'value': '-2.93',
          

## Save the final copy

Set `ready = True` only when the checks and edits are finished.
This checks the edited report and saves it with the masked packet to `public/`.
It replaces existing files with the same IDs, so use this for the working dataset.
Keep released dataset versions unchanged.

Save this notebook with its outputs. These notes are also the feedback;
there is no separate review JSON or feedback file to fill in.


The report is saved as `public/reports/<packet_id>/clean.json`.
For older candidates, the edit cell adds fact IDs only when metric, period, unit and evidence match uniquely. Check the printed references before saving.

In [8]:
ready = True
if ready:
    final_report = Report.model_validate(report)
    final_packet = Packet.model_validate(packet)
    checks = check_report(final_packet, final_report, require_fact_ids=True)
    assert not checks["errors"], checks["errors"]
    assert_masked(report, aliases)
    assert_masked(packet, aliases)
    save(folder / "public/evidence_packets" / (packet["packet_id"] + ".json"), final_packet.model_dump(mode="json"))
    output = folder / "public/reports" / packet["packet_id"] / "clean.json"
    save(output, final_report.model_dump(mode="json"))
    print("Saved:", output)
else:
    print("Not saved yet.")

Saved: /Users/claudia/Desktop/dsa4213/dsa4213 proj/DSA4213_Group11/data/benchmark/public/reports/EP_0008/clean.json
